# Day 2: random baseline submission

Team A | Rishitha Gogineni | 21 November 2026

Build a baseline from the official sample deck and random legal-option selection. Check its callback, run ten battles, then unpack and test the submission archive in a fresh process.

This is the completed Colab run. All local checks passed. The agent was submitted manually afterward; the supplied Kaggle screenshots show **Complete**, first at the initialization rating of **600.0**, then at **490.5**. These snapshots record the same baseline submission, not a model improvement.

The JSON/CSV status outputs were saved before the manual upload and therefore still say `not_submitted`. The later Kaggle evidence is saved separately with the submission record. Python action sampling is seeded; simulator randomness is not verified as seeded. Local tests use individual battles rather than the full Kaggle match harness.


## 1. Environment

Record the runtime and create a separate working folder for this run.


In [1]:
import ast
import csv
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import tarfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

assert platform.system() == 'Linux', 'Use a Linux Colab CPU session.'
assert platform.machine() in ('x86_64', 'amd64', 'aarch64', 'arm64')
assert sys.version_info >= (3, 10)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)

def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
work_dir = Path('/content/ptcg_day2') / RUN_ID
output_dir = work_dir / 'day2_output'
source_dir = output_dir / 'source'
results_dir = output_dir / 'results'
agent_dir = work_dir / 'agent'
for folder in (source_dir, results_dir, agent_dir):
    folder.mkdir(parents=True, exist_ok=False)
environment = {
    'run_id': RUN_ID, 'started_at_utc': utc_now(), 'team': 'GROUP_PROGRAMMING_09_a',
    'model_id': 'A-BASE', 'policy': 'random options, choosing maxCount; official sample deck',
    'python_version': sys.version, 'platform': platform.platform(), 'machine': platform.machine(),
    'simulation_unit': 'individual_battle', 'simulator_seed_verified': False,
    'runtime_dependencies_for_submission': 'Python standard library only',
    'archive': None, 'sdk_files': [], 'local_checks': None,
}
write_json(output_dir / 'environment.json', environment)
print('Python:', sys.version.split()[0], '|', platform.system(), platform.machine())
print('Run:', RUN_ID)


Python: 3.13.18 | Linux x86_64
Run: 20261121T224116Z_1e244d62


## 2. Competition sample

Load the competition ZIP, check the sample deck, and record SDK file hashes. Use the SDK for local testing only.


In [2]:
from google.colab import files

expected_name = 'the-pokemon-company-ptcg-ai-battle-challenge-playground.zip'
archive_path = Path('/content') / expected_name
if not archive_path.exists():
    candidates = sorted(Path('/content').glob('the-pokemon-company-ptcg-ai-battle-challenge-playground*.zip'))
    if len(candidates) == 1:
        archive_path = candidates[0]
    elif len(candidates) > 1:
        raise ValueError('Keep one intended competition ZIP or rename it to ' + expected_name)
    else:
        uploaded = files.upload()
        names = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(names) != 1:
            raise ValueError('Upload exactly one official competition ZIP.')
        archive_path = Path('/content') / names[0]

sdk_dir = work_dir / 'official_sample'
sdk_dir.mkdir()
prefix = 'sample_submission/sample_submission/sample_submission/'
with zipfile.ZipFile(archive_path) as archive:
    members = [name for name in archive.namelist() if name.startswith(prefix) and not name.endswith('/')]
    if not members:
        raise ValueError('Sample package prefix not found. Inspect this ZIP version.')
    for name in members:
        relative = Path(name[len(prefix):])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Unexpected ZIP path.')
        target = sdk_dir / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))

for required in ('main.py', 'deck.csv', 'cg/game.py', 'cg/sim.py'):
    assert (sdk_dir / required).is_file(), required
deck = [int(line) for line in (sdk_dir / 'deck.csv').read_text().splitlines() if line.strip()]
assert len(deck) == 60
environment['archive'] = {'filename': archive_path.name, 'bytes': archive_path.stat().st_size, 'sha256': file_hash(archive_path)}
environment['sdk_files'] = [
    {'path': str(path.relative_to(sdk_dir)), 'sha256': file_hash(path), 'bytes': path.stat().st_size}
    for path in sorted((sdk_dir / 'cg').rglob('*')) if path.is_file()
]
environment['official_sample_agent_sha256'] = file_hash(sdk_dir / 'main.py')
environment['sample_deck_sha256'] = file_hash(sdk_dir / 'deck.csv')
write_json(output_dir / 'environment.json', environment)
print('Sample deck entries:', len(deck))
print('Input ZIP SHA-256:', environment['archive']['sha256'])


Saving the-pokemon-company-ptcg-ai-battle-challenge-playground.zip to the-pokemon-company-ptcg-ai-battle-challenge-playground.zip
Sample deck entries: 60
Input ZIP SHA-256: 6fe557e77ca21b86dea9056d7f4531cbeb328abc1b25aecf9c3a4e2b7d5b2c87


## 3. Baseline agent

Return the sample deck for the initial callback. For other decisions, choose `maxCount` unique option indices, matching the sample policy. The agent uses standard-library imports and reads `deck.csv` from the configured agent path or local package directory.


In [3]:
AGENT_SOURCE = '"""Team A random baseline. Uses the official sample deck and maxCount policy."""\n\nimport os\nimport random\n\n\ndef read_deck(configuration=None):\n    candidates = []\n    raw_path = (configuration or {}).get("__raw_path__")\n    if raw_path:\n        candidates.append(os.path.join(os.path.dirname(os.path.abspath(raw_path)), "deck.csv"))\n    module_path = globals().get("__file__")\n    if module_path:\n        candidates.append(os.path.join(os.path.dirname(os.path.abspath(module_path)), "deck.csv"))\n    candidates.extend(["/kaggle_simulations/agent/deck.csv", "deck.csv"])\n    for path in candidates:\n        if os.path.isfile(path):\n            with open(path, encoding="utf-8") as handle:\n                deck = [int(line.strip()) for line in handle if line.strip()]\n            if len(deck) != 60:\n                raise ValueError("deck.csv must contain exactly 60 card IDs.")\n            return deck\n    raise FileNotFoundError("deck.csv was not found beside the agent or in the Kaggle agent folder.")\n\n\ndef agent(observation, configuration=None):\n    selection = observation.get("select")\n    if selection is None:\n        return read_deck(configuration)\n    options = selection["option"]\n    minimum = selection["minCount"]\n    maximum = selection["maxCount"]\n    if not (0 <= minimum <= maximum <= len(options)):\n        raise ValueError("Invalid option counts in the observation.")\n    return random.sample(range(len(options)), maximum)\n'

ast.parse(AGENT_SOURCE)
(agent_dir / 'main.py').write_text(AGENT_SOURCE, encoding='utf-8')
shutil.copy2(sdk_dir / 'deck.csv', agent_dir / 'deck.csv')
(source_dir / 'baseline.py').write_text(AGENT_SOURCE, encoding='utf-8')
print(AGENT_SOURCE)


"""Team A random baseline. Uses the official sample deck and maxCount policy."""

import os
import random


def read_deck(configuration=None):
    candidates = []
    raw_path = (configuration or {}).get("__raw_path__")
    if raw_path:
        candidates.append(os.path.join(os.path.dirname(os.path.abspath(raw_path)), "deck.csv"))
    module_path = globals().get("__file__")
    if module_path:
        candidates.append(os.path.join(os.path.dirname(os.path.abspath(module_path)), "deck.csv"))
    candidates.extend(["/kaggle_simulations/agent/deck.csv", "deck.csv"])
    for path in candidates:
        if os.path.isfile(path):
            with open(path, encoding="utf-8") as handle:
                deck = [int(line.strip()) for line in handle if line.strip()]
            if len(deck) != 60:
                raise ValueError("deck.csv must contain exactly 60 card IDs.")
            return deck
    raise FileNotFoundError("deck.csv was not found beside the agent or in the Kaggle agent folder.

## 4. Battle runner

Call the actual agent functions for deck selection and each decision. Check response counts and indices, record errors, and release the engine after each battle. This runner handles individual battles, not best-of-three matches or RL training trajectories.


In [4]:
RUNNER_SOURCE = '"""Reusable direct-SDK individual-battle runner. SDK must be importable as cg."""\n\nimport random\nimport time\nimport traceback\n\n\ndef check_action(action, selection):\n    if not isinstance(action, list) or not all(type(index) is int for index in action):\n        raise TypeError("The agent must return a list of integer option indices.")\n    if not selection["minCount"] <= len(action) <= selection["maxCount"]:\n        raise ValueError("Action length violates selection counts.")\n    if len(set(action)) != len(action):\n        raise ValueError("Repeated option indices.")\n    if any(index < 0 or index >= len(selection["option"]) for index in action):\n        raise ValueError("Option index out of range.")\n\n\ndef run_battle(agents, seed, max_steps=10000, reverse_player=False):\n    from cg.game import battle_start, battle_select, battle_finish\n    from cg.sim import Battle\n\n    random.seed(seed)\n    started = time.perf_counter()\n    record = {\n        "python_seed": seed, "unit": "individual_battle", "status": "running",\n        "finished": False, "result_code": None, "steps": 0,\n        "reverse_player": reverse_player, "seconds": None,\n        "max_decision_ms": 0.0, "selection_types": [],\n        "invalid_responses": 0, "engine_rejections": 0,\n        "error_type": "", "error_message": "", "traceback": "", "cleanup_error": "",\n    }\n    seen_types = set()\n    try:\n        initial = {"select": None, "logs": [], "current": None, "search_begin_input": None}\n        decks = [callback(dict(initial)) for callback in agents]\n        for deck in decks:\n            if not isinstance(deck, list) or len(deck) != 60 or not all(type(card) is int for card in deck):\n                raise ValueError("Initial callback did not return 60 integer card IDs.")\n        observation, start = battle_start(decks[0], decks[1], reverse_player)\n        if observation is None:\n            raise RuntimeError(f"Battle start error: player={start.errorPlayer}, type={start.errorType}")\n        for step in range(max_steps + 1):\n            current = observation.get("current")\n            if current is None:\n                raise ValueError("Missing current state in direct battle API.")\n            result_code = current["result"]\n            if result_code != -1:\n                record.update(status="passed", finished=True, result_code=result_code)\n                break\n            if step == max_steps:\n                record.update(status="step_limit", error_type="StepLimit", error_message=f"No terminal result after {max_steps} selections.")\n                break\n            player = current["yourIndex"]\n            if player not in (0, 1):\n                raise ValueError("Unexpected acting player.")\n            selection = observation.get("select")\n            if selection is None:\n                raise ValueError("Missing selection during the battle.")\n            seen_types.add(selection["type"])\n            t0 = time.perf_counter()\n            action = agents[player](observation)\n            record["max_decision_ms"] = max(record["max_decision_ms"], (time.perf_counter() - t0) * 1000)\n            try:\n                check_action(action, selection)\n            except Exception:\n                record["invalid_responses"] += 1\n                raise\n            try:\n                observation = battle_select(action)\n            except Exception:\n                record["engine_rejections"] += 1\n                raise\n            record["steps"] = step + 1\n    except Exception as error:\n        record.update(status="error", error_type=type(error).__name__, error_message=str(error), traceback=traceback.format_exc())\n    finally:\n        try:\n            if Battle.battle_ptr:\n                battle_finish()\n        except Exception as error:\n            record["cleanup_error"] = f"{type(error).__name__}: {error}"\n            record["status"] = "cleanup_error"\n        finally:\n            Battle.battle_ptr = None\n            record["seconds"] = round(time.perf_counter() - started, 6)\n            record["max_decision_ms"] = round(record["max_decision_ms"], 6)\n            record["selection_types"] = sorted(seen_types)\n    return record\n'

ast.parse(RUNNER_SOURCE)
(source_dir / 'simulator.py').write_text(RUNNER_SOURCE, encoding='utf-8')
print('Saved reusable runner:', source_dir / 'simulator.py')


Saved reusable runner: /content/ptcg_day2/20261121T224116Z_1e244d62/day2_output/source/simulator.py


## 5. Callback checks and ten battles

Check initial deck loading, single/multiple choices, optional choices and empty choices. Load the agent from an unrelated working directory without `__file__`, then run ten random-versus-random battles in a separate process. These are local compatibility checks, not the full hosted Kaggle validation.


In [5]:
WORKER_SOURCE = '"""Run callback checks and SDK battles in an isolated Python process."""\n\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport traceback\n\n\ndef save_json(path, value):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef load_callback(path):\n    # Match the inspected Kaggle loader\'s exec-and-last-callable behavior.\n    namespace = {}\n    exec(compile(path.read_text(), str(path), "exec"), namespace)\n    functions = [value for value in namespace.values() if callable(value)]\n    if not functions or functions[-1].__name__ != "agent":\n        raise ValueError("The last callable in main.py must be agent.")\n    callback = functions[-1]\n    configuration = {"__raw_path__": str(path)}\n    return lambda observation: callback(observation, configuration)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--agent-dir", required=True)\n    parser.add_argument("--sdk-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--games", type=int, default=10)\n    parser.add_argument("--seed-base", type=int, default=100)\n    args = parser.parse_args()\n    agent_dir = Path(args.agent_dir).resolve()\n    out = Path(args.output_dir).resolve()\n    out.mkdir(parents=True, exist_ok=True)\n    sys.path.insert(0, str(Path(args.sdk_dir).resolve()))\n    from simulator import check_action, run_battle\n\n    report = {\n        "status": "running", "all_passed": False, "unit": "individual_battle",\n        "loader_check": "exec without __file__; last callable; configured agent path",\n        "contract_checks": [], "battles": [], "error": None,\n        "scope": "Direct SDK and local loader checks; not the full Kaggle harness or a leaderboard evaluation.",\n    }\n    save_json(out / "report.json", report)\n    try:\n        agents = [load_callback(agent_dir / "main.py") for _ in range(2)]\n        initial = {"select": None, "logs": [], "current": None, "search_begin_input": None}\n        returned_deck = agents[0](initial)\n        expected_deck = [int(line) for line in (agent_dir / "deck.csv").read_text().splitlines() if line.strip()]\n        assert returned_deck == expected_deck and len(returned_deck) == 60\n        report["contract_checks"].append({"case": "initial_deck_from_unrelated_working_directory", "passed": True})\n        cases = [\n            ("single", {"minCount": 1, "maxCount": 1, "option": [{}, {}]}),\n            ("multiple", {"minCount": 2, "maxCount": 3, "option": [{}, {}, {}, {}, {}]}),\n            ("zero_options", {"minCount": 0, "maxCount": 0, "option": []}),\n            ("optional", {"minCount": 0, "maxCount": 2, "option": [{}, {}, {}]}),\n        ]\n        for name, selection in cases:\n            check_action(agents[0]({"select": selection}), selection)\n            report["contract_checks"].append({"case": name, "passed": True})\n        save_json(out / "report.json", report)\n        for index in range(args.games):\n            record = run_battle(agents, seed=args.seed_base + index, reverse_player=bool(index % 2))\n            record["battle_id"] = index + 1\n            report["battles"].append(record)\n            save_json(out / "report.json", report)\n            print({key: record[key] for key in ("battle_id", "status", "steps", "seconds", "invalid_responses", "engine_rejections")}, flush=True)\n        report["all_passed"] = (\n            len(report["battles"]) == args.games\n            and all(row["finished"] and row["status"] == "passed" for row in report["battles"])\n            and all(row["passed"] for row in report["contract_checks"])\n        )\n        report["status"] = "passed" if report["all_passed"] else "failed"\n    except Exception as error:\n        report.update(status="error", error={"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()})\n    finally:\n        save_json(out / "report.json", report)\n        fields = ["battle_id", "python_seed", "status", "finished", "result_code", "steps", "seconds", "max_decision_ms", "reverse_player", "invalid_responses", "engine_rejections", "error_type", "error_message", "cleanup_error"]\n        with (out / "battle_results.csv").open("w", newline="", encoding="utf-8") as handle:\n            writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n            writer.writeheader()\n            writer.writerows(report["battles"])\n    print("Local checks:", report["status"], flush=True)\n    return 0 if report["all_passed"] else 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

worker_path = source_dir / 'check_agent.py'
ast.parse(WORKER_SOURCE)
worker_path.write_text(WORKER_SOURCE, encoding='utf-8')

def run_worker(label, package_directory, games, seed_base):
    test_dir = results_dir / label
    test_dir.mkdir(parents=True, exist_ok=False)
    working_directory = work_dir / (label + '_working_directory')
    working_directory.mkdir()
    command = [sys.executable, str(worker_path), '--agent-dir', str(package_directory),
               '--sdk-dir', str(sdk_dir), '--output-dir', str(test_dir),
               '--games', str(games), '--seed-base', str(seed_base)]
    try:
        completed = subprocess.run(command, cwd=working_directory, capture_output=True, text=True, timeout=600)
        process = {'returncode': completed.returncode, 'timed_out': False}
        stdout, stderr = completed.stdout, completed.stderr
    except subprocess.TimeoutExpired as error:
        process = {'returncode': None, 'timed_out': True}
        stdout = error.stdout or ''
        stderr = error.stderr or ''
        if isinstance(stdout, bytes): stdout = stdout.decode(errors='replace')
        if isinstance(stderr, bytes): stderr = stderr.decode(errors='replace')
    (test_dir / 'stdout.txt').write_text(stdout, encoding='utf-8')
    (test_dir / 'stderr.txt').write_text(stderr, encoding='utf-8')
    write_json(test_dir / 'process.json', process)
    report_path = test_dir / 'report.json'
    report = json.loads(report_path.read_text()) if report_path.exists() else {'all_passed': False, 'status': 'no_report'}
    passed = process['returncode'] == 0 and report.get('all_passed') is True
    print(stdout[-5000:])
    if not passed:
        print('Checks failed. Saved process/report files contain details.')
        print(stderr[-2000:])
    return passed, report

local_passed, local_report = run_worker('local_checks', agent_dir, games=10, seed_base=100)
environment['local_checks'] = {'passed': local_passed, 'games_requested': 10, 'report': 'results/local_checks/report.json'}
write_json(output_dir / 'environment.json', environment)


{'battle_id': 1, 'status': 'passed', 'steps': 36, 'seconds': 0.010039, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 2, 'status': 'passed', 'steps': 32, 'seconds': 0.007815, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 3, 'status': 'passed', 'steps': 119, 'seconds': 0.033364, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 4, 'status': 'passed', 'steps': 39, 'seconds': 0.014793, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 5, 'status': 'passed', 'steps': 22, 'seconds': 0.008372, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 6, 'status': 'passed', 'steps': 64, 'seconds': 0.025155, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 7, 'status': 'passed', 'steps': 60, 'seconds': 0.014935, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 8, 'status': 'passed', 'steps': 36, 'seconds': 0.010617, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 9, 'status': 'passed', 'steps': 3

## 6. Submission archive

Package `main.py` and `deck.csv` at the archive root. Verify file hashes and run two more battles from a fresh extraction. Include the archive in the download only if these checks pass.


In [6]:
submission_path = output_dir / 'submission.tar.gz'
roundtrip_passed = False
roundtrip_report = None
package_manifest = {'built': False, 'local_package_ready': False, 'kaggle_validation_status': 'not_submitted'}

if local_passed:
    with tarfile.open(submission_path, 'w:gz') as archive:
        for name in ('main.py', 'deck.csv'):
            archive.add(agent_dir / name, arcname=name, recursive=False)
    extracted_dir = work_dir / 'unpacked_submission'
    extracted_dir.mkdir()
    with tarfile.open(submission_path, 'r:gz') as archive:
        members = archive.getmembers()
        assert len(members) == 2 and {item.name for item in members} == {'main.py', 'deck.csv'}
        for item in members:
            assert item.isfile() and not item.issym() and not item.islnk()
            content = archive.extractfile(item).read()
            assert content == (agent_dir / item.name).read_bytes()
            (extracted_dir / item.name).write_bytes(content)
    assert submission_path.stat().st_size <= int(197.7 * 1024 * 1024)
    package_manifest.update(
        built=True, members=['main.py', 'deck.csv'], bytes=submission_path.stat().st_size,
        sha256=file_hash(submission_path),
        files={name: file_hash(extracted_dir / name) for name in ('main.py', 'deck.csv')},
    )
    roundtrip_passed, roundtrip_report = run_worker('archive_checks', extracted_dir, games=2, seed_base=200)
    package_manifest['local_package_ready'] = roundtrip_passed
    if not roundtrip_passed:
        # Keep the identity in the manifest, but do not download a failed package.
        submission_path.unlink()
else:
    print('Archive not built because the initial checks did not pass.')

write_json(output_dir / 'package_manifest.json', package_manifest)
environment['archive_checks'] = {'passed': roundtrip_passed, 'games_requested': 2, 'report': 'results/archive_checks/report.json'}
write_json(output_dir / 'environment.json', environment)
print('Local package ready:', package_manifest['local_package_ready'])
if package_manifest['local_package_ready']:
    print('Archive members:', package_manifest['members'])
    print('Archive bytes:', package_manifest['bytes'])
    print('Archive SHA-256:', package_manifest['sha256'])


{'battle_id': 1, 'status': 'passed', 'steps': 36, 'seconds': 0.013966, 'invalid_responses': 0, 'engine_rejections': 0}
{'battle_id': 2, 'status': 'passed', 'steps': 31, 'seconds': 0.012438, 'invalid_responses': 0, 'engine_rejections': 0}
Local checks: passed

Local package ready: True
Archive members: ['main.py', 'deck.csv']
Archive bytes: 892
Archive SHA-256: c889d1b602458c63f6aad64e691acc47972517bdb3a9ccd93ea6b87635075c96


## 7. Pre-submission record

Save the archive identity and local results. The following status was recorded before uploading to Kaggle; official submission details are recorded separately afterward.


In [7]:
fields = ['team', 'member', 'model_id', 'run_id', 'artifact_sha256', 'local_status',
          'submission_id', 'submitted_at_utc', 'kaggle_validation_status', 'initial_rating',
          'rating_observed_at_utc', 'rating_uncertainty', 'notes']
row = {
    'team': 'GROUP_PROGRAMMING_09_a', 'member': 'Rishitha Gogineni', 'model_id': 'A-BASE',
    'run_id': RUN_ID, 'artifact_sha256': package_manifest.get('sha256', ''),
    'local_status': 'package_ready' if package_manifest['local_package_ready'] else 'local_checks_failed',
    'submission_id': '', 'submitted_at_utc': '', 'kaggle_validation_status': 'not_submitted',
    'initial_rating': '', 'rating_observed_at_utc': '', 'rating_uncertainty': '',
    'notes': 'Random policy and official sample deck; individual-battle checks; not a trained model.',
}
with (output_dir / 'baseline_submission_record.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerow(row)

status = {
    'run_id': RUN_ID, 'local_checks_passed': local_passed,
    'archive_roundtrip_checks_passed': roundtrip_passed,
    'local_package_ready': package_manifest['local_package_ready'],
    'kaggle_submitted': False, 'kaggle_accepted': False, 'official_rating': None,
    'note': 'These flags record local checks only. Update official evidence after manual submission.',
}
write_json(output_dir / 'day2_status.json', status)
(output_dir / 'README.txt').write_text(
    'DAY 2 BASELINE OUTPUTS\n\n'
    'If local_package_ready is true, upload submission.tar.gz to the Pokémon Playground.\n'
    'Do not upload this entire results ZIP. Check the Kaggle team name first.\n'
    'The archive contains the random policy and official sample deck. No trained model is included.\n'
    'Local checks do not replace official Kaggle validation or a leaderboard score.\n'
    'Record the submission ID, status and initial rating when available.\n'
    'source/baseline.py and source/simulator.py are reusable code exports; the runner needs the local official SDK.\n'
    'results/ contains callback checks, game summaries, process status and errors.\n'
    'The environment and manifest identify the exact inputs and archive tested.\n', encoding='utf-8')
print(json.dumps(status, indent=2))


{
  "run_id": "20261121T224116Z_1e244d62",
  "local_checks_passed": true,
  "archive_roundtrip_checks_passed": true,
  "local_package_ready": true,
  "kaggle_submitted": false,
  "kaggle_accepted": false,
  "official_rating": null,
  "note": "These flags record local checks only. Update official evidence after manual submission."
}


## 8. Download outputs

Download the output ZIP. Its inner `submission.tar.gz` is the Kaggle upload file. Keep the source exports and test records for integration and reproducibility.


In [8]:
bundle_path = work_dir / f'Day2_{RUN_ID}_outputs.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output_dir.rglob('*')):
        if path.is_file() and path.suffix != '.tmp':
            bundle.write(path, arcname=str(path.relative_to(output_dir)))
print('Download:', bundle_path.name)
if package_manifest['local_package_ready']:
    print('Extract submission.tar.gz from this ZIP for Kaggle.')
else:
    print('No submission archive included. Share the notebook and diagnostics for review.')
files.download(str(bundle_path))


Download: Day2_20261121T224116Z_1e244d62_outputs.zip
Extract submission.tar.gz from this ZIP for Kaggle.


## Results

| Check | Result |
| --- | --- |
| Initial deck and option-response checks | Passed |
| Local battles | 10/10 passed |
| Battles after unpacking the archive | 2/2 passed |
| Invalid responses and engine rejections | 0 |
| Submission archive | 892 bytes; root-level `main.py` and `deck.csv` |
| Kaggle status in supplied screenshot | Complete |
| Initialization rating shown | 600.0 |
| Later rating shown | 490.5 |

The screenshot records the first baseline result. The submission ID, uncertainty, exact submission time and actual team name are not visible in that image. The 600.0 snapshot is the initialization point; a later screenshot shows 490.5. Ratings change as episodes are played, so retain the sequence and do not treat initialization as measured improvement.

Next, use the integrated baseline and runner source in the repository, keep tracking submission evidence, and begin the assigned linear/MLP model work once the feature interface and training data are ready. The random baseline is not one of those learned architectures.

References: [competition](https://www.kaggle.com/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground), [simulator documentation](https://matsuoinstitute.github.io/cabt/), [Kaggle agent loader](https://github.com/Kaggle/kaggle-environments/blob/master/kaggle_environments/agent.py).
